# YearPredictionMSD: Full-Strength Replication of the APLR Paper

Goal: replicate Table 15 of von Ottenbreit & De Bin (2024), *Automatic piecewise
linear regression*, Computational Statistics 39, 1867-1907. Once the numbers are
matched, the results can be used to look for gaps and improvements.

This notebook is meant to run on a multi-core machine (tested design target: 8 cores,
16 GB RAM) or on Colab. It differs from the earlier single-core version in three ways:

- Random Forest and APLR run at full strength instead of the reduced settings.
- APLR uses the paper's hyperparameter grid through `APLRTuner`, then a final
  cross-validated refit.
- MARS and gamboost/glmboost are run in R (they have no working Python install).

Every model saves its result to disk as soon as it finishes, so a long run can be
resumed after an interruption. Run the cells top to bottom. Run one heavy model at a time.

Set the environment variable `QUICK_TEST=1` before starting Jupyter to smoke-test the
whole notebook in a couple of minutes on a small subsample (numbers from a quick test
mean nothing; they only check that the code runs).

## 1. Setup and configuration

In [1]:
import os, json, time, subprocess, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from aplr import APLRRegressor, APLRTuner

# ---- Paths: edit DATA_PATH to where YearPredictionMSD.csv lives ----
# Local Windows example: r"C:\Users\you\Downloads\YearPredictionMSD.csv"
# Colab: mount Drive first (uncomment the two lines below) and point to the file in Drive.
# from google.colab import drive; drive.mount("/content/drive")
DATA_PATH = os.environ.get("MSD_CSV", "YearPredictionMSD.csv")

QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
RESULTS_DIR = "results_quick" if QUICK_TEST else "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS_PATH = os.path.join(RESULTS_DIR, "results.json")

N_JOBS = os.cpu_count() or 1
RANDOM_STATE = 42
print(f"CPU cores available: {N_JOBS} | QUICK_TEST={QUICK_TEST}")

CPU cores available: 16 | QUICK_TEST=False


## 2. Load the data and apply the official split

The paper uses the dataset's official split on the original row order: first 463,715
rows for training, last 51,630 for testing (about 90/10). No shuffling, because songs by
the same artist are stored together and a random split would leak artist information.

In [2]:
df = pd.read_csv(DATA_PATH)
assert df.shape == (515345, 91), f"Unexpected shape {df.shape}"
assert df.isna().sum().sum() == 0

N_TRAIN = 463_715
train_df, test_df = df.iloc[:N_TRAIN], df.iloc[N_TRAIN:]
X_train = train_df.drop(columns="Year").values.astype(np.float64)
y_train = train_df["Year"].values.astype(np.float64)
X_test = test_df.drop(columns="Year").values.astype(np.float64)
y_test = test_df["Year"].values.astype(np.float64)

if QUICK_TEST:  # tiny subsample, only to check the code runs
    X_train, y_train = X_train[:8000], y_train[:8000]
    X_test, y_test = X_test[:2000], y_test[:2000]

del df, train_df, test_df
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (463715, 90) Test: (51630, 90)


## 3. Paper's reported results (Table 15)

In [3]:
paper = pd.DataFrame(
    {"MSE": [79.93, 80.06, 83.60, 86.19, 87.01, 90.72],
     "R2": [0.3213, 0.3208, 0.2947, 0.2686, 0.2615, 0.2297]},
    index=["APLR", "LightGBM", "Random Forest", "MARS", "gamboost", "glmboost"],
)
paper

,MSE,R2
APLR,79.93,0.3213
LightGBM,80.06,0.3208
Random Forest,83.60,0.2947
MARS,86.19,0.2686
gamboost,87.01,0.2615
glmboost,90.72,0.2297


## 4. Helpers

`run_once` skips a model whose result is already saved, evaluates on the test set,
writes the result to `results.json`, and saves test predictions as a `.npy` file for
later gap analysis.

In [4]:
def load_results():
    if os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH) as f:
            return json.load(f)
    return {}

def save_result(name, mse, r2, fit_time, params):
    res = load_results()
    res[name] = {"MSE": mse, "R2": r2, "fit_time_s": fit_time, "params": params}
    with open(RESULTS_PATH, "w") as f:
        json.dump(res, f, indent=2)

def run_once(name, fit_predict):
    # fit_predict() must return (y_pred_on_test, params_dict). Skipped if already saved.
    if name in load_results():
        r = load_results()[name]
        print(f"{name}: already done, MSE={r['MSE']:.4f} R2={r['R2']:.4f} (delete from results.json to rerun)")
        return
    t0 = time.time()
    pred, params = fit_predict()
    fit_time = time.time() - t0
    mse, r2 = mean_squared_error(y_test, pred), r2_score(y_test, pred)
    np.save(os.path.join(RESULTS_DIR, f"pred_{name.replace(' ', '_')}.npy"), pred)
    save_result(name, mse, r2, fit_time, params)
    print(f"{name}: MSE={mse:.4f} R2={r2:.4f} fit_time={fit_time/60:.1f} min")

## 5. Linear baseline (glmboost analogue)

Ordinary least squares on standardised features. The real `glmboost` is run in R in
section 10; this is a fast Python stand-in that already matches it closely.

In [5]:
def fit_linear():
    sc = StandardScaler().fit(X_train)
    m = LinearRegression().fit(sc.transform(X_train), y_train)
    return m.predict(sc.transform(X_test)), {}

run_once("Linear (glmboost analogue)", fit_linear)

Linear (glmboost analogue): MSE=90.4432 R2=0.2320 fit_time=0.1 min


## 6. LightGBM

`num_leaves` is tuned over a grid (the paper reports 12 as best) using the last 10% of
the training rows as a validation set. The split is positional, not random, so the
validation set follows the same no-leakage logic as the official split. Early stopping
picks the number of rounds, then the model is refit on the full training set.

In [6]:
def fit_lightgbm():
    n_val = int(len(X_train) * 0.10)
    Xf, yf, Xv, yv = X_train[:-n_val], y_train[:-n_val], X_train[-n_val:], y_train[-n_val:]
    grid = [6, 12, 24, 48, 128] if not QUICK_TEST else [12, 24]
    best = None
    for nl in grid:
        m = lgb.LGBMRegressor(n_estimators=5000, num_leaves=nl, learning_rate=0.05,
                              n_jobs=N_JOBS, verbosity=-1, random_state=RANDOM_STATE)
        m.fit(Xf, yf, eval_set=[(Xv, yv)], eval_metric="l2",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        val_mse = m.best_score_["valid_0"]["l2"]
        print(f"  num_leaves={nl}: val MSE={val_mse:.3f}, rounds={m.best_iteration_}")
        if best is None or val_mse < best[0]:
            best = (val_mse, nl, m.best_iteration_)
    _, nl, rounds = best
    final = lgb.LGBMRegressor(n_estimators=int(rounds * 1.1), num_leaves=nl, learning_rate=0.05,
                              n_jobs=N_JOBS, verbosity=-1, random_state=RANDOM_STATE)
    final.fit(X_train, y_train)
    return final.predict(X_test), {"num_leaves": nl, "n_estimators": int(rounds * 1.1), "learning_rate": 0.05}

run_once("LightGBM", fit_lightgbm)

c:\Users\232006\Downloads\Local\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  num_leaves=6: val MSE=76.882, rounds=4999


c:\Users\232006\Downloads\Local\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  num_leaves=12: val MSE=76.608, rounds=3760


c:\Users\232006\Downloads\Local\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  num_leaves=24: val MSE=76.326, rounds=3445


c:\Users\232006\Downloads\Local\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  num_leaves=48: val MSE=76.234, rounds=2849


c:\Users\232006\Downloads\Local\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  num_leaves=128: val MSE=76.337, rounds=734
LightGBM: MSE=79.5286 R2=0.3247 fit_time=7.9 min


## 7. Random Forest (full strength)

The earlier run used 40 shallow trees with 25% subsampling per tree. This run uses
200 deeper trees on full bootstrap samples. Memory is the constraint: if you get an
out-of-memory error, lower `n_estimators` or raise `min_samples_leaf`, close other
programs, and rerun. `max_features=0.33` follows the common regression default of
about one third of the features.

In [7]:
RF_TREES = 200 if not QUICK_TEST else 20

def fit_rf():
    rf = RandomForestRegressor(n_estimators=RF_TREES, min_samples_leaf=5, max_features=0.33,
                               n_jobs=N_JOBS, random_state=RANDOM_STATE)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    del rf
    return pred, {"n_estimators": RF_TREES, "min_samples_leaf": 5, "max_features": 0.33}

run_once("Random Forest", fit_rf)

Random Forest: MSE=84.0091 R2=0.2866 fit_time=5.5 min


## 8. APLR (the paper's grid, then a cross-validated refit)

The paper tunes `max_interaction_level` in {0, 1, 2, 100} and `min_observations_in_split`
in {20, 100, 500}, with `v=0.5` and up to `m=3500` boosting steps. The grid is searched
with a single validation split (`validation_ratio=0.2`, the faster option in the
package's own example), then the best setting is refit with 5-fold cross-validation for
the final model. This is the slowest cell: expect it to take hours on 8 cores. Because it
saves its result only at the end, run it when the machine can stay on. If it is too slow,
shrink the grid or lower `m` (for example 1500) and note the change.

In [ ]:
def fit_aplr():
    params = {
        "m": [3500 if not QUICK_TEST else 200],
        "v": [0.5],
        "max_interaction_level": [0, 1, 2, 100] if not QUICK_TEST else [1, 2],
        "min_observations_in_split": [20, 100, 500] if not QUICK_TEST else [100],
        "loss_function": ["mse"],
        "link_function": ["identity"],
        "validation_ratio": [0.2],
        "n_jobs": [0],           # 0 = use all cores
        "random_state": [RANDOM_STATE],
        "verbosity": [1],
    }
    tuner = APLRTuner(parameters=params, is_regressor=True, sequential_tuning=False)
    tuner.fit(X=X_train, y=y_train)
    best = tuner.get_best_estimator()
    best_params = {k: best.get_params()[k] for k in ("max_interaction_level", "min_observations_in_split", "m", "v")}
    print("Best grid point:", best_params)
    best.set_params(validation_ratio=np.nan, cv_folds=5)   # final refit with 5-fold CV
    best.fit(X_train, y_train)
    return best.predict(X_test), best_params

run_once("APLR", fit_aplr)

Model terms: 1887. Terms available in final boosting step: 90.
Model terms: 1878. Terms available in final boosting step: 90.


## 9. Results so far vs the paper

In [ ]:
def comparison():
    res = load_results()
    rows = []
    name_map = {"APLR": "APLR", "LightGBM": "LightGBM", "Random Forest": "Random Forest",
                "Linear (glmboost analogue)": "glmboost", "MARS": "MARS",
                "gamboost": "gamboost", "glmboost (R)": "glmboost"}
    for ours, theirs in name_map.items():
        if ours in res:
            rows.append({"Model": ours, "Paper MSE": paper.loc[theirs, "MSE"], "Our MSE": res[ours]["MSE"],
                         "MSE gap": res[ours]["MSE"] - paper.loc[theirs, "MSE"],
                         "Paper R2": paper.loc[theirs, "R2"], "Our R2": res[ours]["R2"]})
    return pd.DataFrame(rows).set_index("Model").round(4)

table = comparison()
table

In [ ]:
ax = table[["Paper MSE", "Our MSE"]].plot(kind="bar", figsize=(9, 4))
ax.set_ylabel("Test MSE (lower is better)")
ax.set_title("Paper vs this replication")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

## 10. MARS, gamboost and glmboost in R

These three have no working Python install, so they run in R. This section needs R
(https://cran.r-project.org) with the packages `earth` and `mboost`; the code installs
them if missing. It calls `Rscript` from Python and reads the results back. If `Rscript`
is not found the cell just prints a message and everything else still works.

Settings: MARS uses `degree=4` (the paper's best-found maximum degree). The boosting
models use `nu=0.1` with the number of boosting steps chosen by 5-fold `cvrisk`
(capped at 3000 steps). On Windows, install R from CRAN and make sure `Rscript` is on
your PATH, or set `RSCRIPT` below to its full path.

In [ ]:
RSCRIPT = shutil.which("Rscript")  # or set e.g. r"C:\Program Files\R\R-4.4.1\bin\Rscript.exe"

R_CODE = '''
args <- commandArgs(trailingOnly = TRUE)
csv <- args[1]; out <- args[2]; quick <- args[3] == "1"
for (p in c("earth", "mboost")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p, repos = "https://cloud.r-project.org")
library(earth); library(mboost)
d <- read.csv(csv)
n_train <- 463715
train <- d[1:n_train, ]; test <- d[(n_train + 1):nrow(d), ]
if (quick) { train <- train[1:8000, ]; test <- test[1:2000, ] }
mse <- function(p) mean((test$Year - p)^2)
r2 <- function(p) 1 - sum((test$Year - p)^2) / sum((test$Year - mean(test$Year))^2)
res <- list()
t0 <- Sys.time()
m <- earth(Year ~ ., data = train, degree = 4)
p <- as.numeric(predict(m, test))
res$MARS <- list(MSE = mse(p), R2 = r2(p), fit_time_s = as.numeric(difftime(Sys.time(), t0, units = "secs")))
cap <- if (quick) 100 else 3000
for (nm in c("glmboost", "gamboost")) {
  t0 <- Sys.time()
  m <- if (nm == "glmboost") glmboost(Year ~ ., data = train, control = boost_control(mstop = cap, nu = 0.1))
       else gamboost(Year ~ ., data = train, control = boost_control(mstop = cap, nu = 0.1))
  cv <- cvrisk(m, folds = cv(model.weights(m), type = "kfold", B = 5))
  m[mstop(cv)]
  p <- as.numeric(predict(m, test))
  res[[nm]] <- list(MSE = mse(p), R2 = r2(p), fit_time_s = as.numeric(difftime(Sys.time(), t0, units = "secs")), mstop = mstop(cv))
}
writeLines(paste0("{", paste(sprintf('"%s": {%s}', names(res), sapply(res, function(x) paste(sprintf('"%s": %s', names(x), unlist(x)), collapse = ", "))), collapse = ", "), "}"), out)
'''

def run_r_models():
    if RSCRIPT is None:
        print("Rscript not found: skipping MARS / gamboost / glmboost. Install R and rerun this cell.")
        return
    csv_abs = os.path.abspath(DATA_PATH)
    out_json = os.path.join(RESULTS_DIR, "r_results.json")
    r_file = os.path.join(RESULTS_DIR, "run_models.R")
    with open(r_file, "w") as f:
        f.write(R_CODE)
    subprocess.run([RSCRIPT, r_file, csv_abs, out_json, "1" if QUICK_TEST else "0"], check=True)
    with open(out_json) as f:
        r_res = json.load(f)
    key = {"MARS": "MARS", "gamboost": "gamboost", "glmboost": "glmboost (R)"}
    for k, v in r_res.items():
        save_result(key[k], v["MSE"], v["R2"], v["fit_time_s"], {kk: vv for kk, vv in v.items() if kk == "mstop"})
        print(f"{key[k]}: MSE={v['MSE']:.4f} R2={v['R2']:.4f}")

run_r_models()

## 11. Final comparison with the paper

In [ ]:
final_table = comparison()
final_table

In [ ]:
ax = final_table[["Paper MSE", "Our MSE"]].plot(kind="bar", figsize=(10, 4))
ax.set_ylabel("Test MSE (lower is better)")
ax.set_title("Paper vs this replication (all available models)")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

## 12. Reading the result

- A model counts as replicated when its MSE is close to the paper's (a gap of a few
  tenths of an MSE point is normal run-to-run and implementation noise).
- If APLR or Random Forest is still noticeably worse than the paper, check the saved
  parameters in `results/results.json` against the paper's reported settings and raise
  the budget (more trees, larger `m`, full grid) before drawing conclusions.
- Once the table matches, the saved test predictions in `results/pred_*.npy` can be
  used for the gap analysis (per-decade errors, residual patterns, ensembling, repeated
  runs for confidence intervals).